---   
 <img align="left" width="75" height="75"  src="https://upload.wikimedia.org/wikipedia/en/c/c8/University_of_the_Punjab_logo.png">

<h1 align="center">Department of Data Science</h1>
<h1 align="center">Course: Generative and Agentic AI</h1>

---
<h3><div align="right">Instructor: Muhammad Arif Butt, Ph.D.</div></h3>    

<br><br>
<h1 align="center">Lec-26: Fine-Tuning TinyLlama using <b>Unsloth</b></h1>

## **Learning Agenda of This Notebook**

1. Check your Runtime and Install Dependencies (installation needed only on Google Colab)
2. Dataset Curation
3. Configuration of Model, Dataset, LoRA Adapters, and Training Hyperparameters
4. Load Base Model + Tokenizer
5. Prepare Dataset
6. Configure and Inject LoRA Adapters
7. Configure Training Parameters
8. Train the Model (needs an NVIDIA GPU: your own machine or Google Colab)
9. Training Loss Plot
10. Save the Adapter Weights, Push to HF and Do Inference (`arif-butt/tinyllama-unsloth-lora-adapter`)
11. Merge Adapter Weights, Push to HF and Do Inference (`arif-butt/tinyllama-unsloth-merged`)
12. Generate GGUF and Push to HF and Do Inference via Ollama and Llama.cpp (`arif-butt/tinyllama-unsloth-gguf`)
13. Inference with the Merged Model from the Hub: `pipeline()` vs `generate()`
14. Windows Quick Guide: Ollama and llama.cpp

# <span style='background :lightgreen' >Why Unsloth?</span>

| | Standard TRL | Unsloth + TRL |
|---|---|---|
| Training speed | Baseline | 2–5× faster |
| Memory usage | High | ~60% less VRAM |
| LoRA injection | via PEFT | Unsloth optimized kernels |
| GGUF export | Manual (llama.cpp) | Built-in `save_pretrained_gguf()` |
| Accuracy | Same | Same |

Unsloth replaces `transformers` for model loading and `peft` for LoRA injection — but `trl.SFTTrainer` is still used for training, exactly as before.


# <span style='background :lightgreen' >1. Check your Runtime and Install Dependencies</span>

This notebook runs **both on your own computer and on Google Colab**. No NVIDIA GPU on your computer? Click this button to open this notebook in Google Colab:

<a href="https://colab.research.google.com/github/arifpucit/Generative-and-Agentic-AI/blob/main/Notebooks/Lec_26_%28FineTuning_Unsloth%29.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


| Where | What to do |
|---|---|
| **Your computer** | Create the environment once with `uv sync` (from the repository root) and open the notebook with the project kernel. **Skip cell 1b** — the virtual environment already has every package. |
| **Google Colab** | Click the **Open in Colab** button above (or the link printed by cell 1a). In Colab select `Runtime > Change runtime type > T4 GPU`, then run cell **1a** and cell **1b** (1b installs the packages), and then the rest of the notebook. |

- Every cell runs on CPU, Mac or GPU **except the training cell**, which needs an **NVIDIA (CUDA) GPU with at least 6 GB** of memory.
- On a machine without such a GPU, the training cell prints a message and is skipped. The later cells then use the fine-tuned model that is already published on the Hugging Face Hub, so you can still follow the rest of the notebook.

## a. Check your runtime (always run this cell first)

In [ ]:
# ── Run this cell FIRST ─────────────────────────────────────────────────────────────────────
# It tells you whether the TRAINING cell of this notebook can run on this machine or whether you need Google Colab.
# All other cells run everywhere (CPU, Mac, Colab); only the training cell needs an NVIDIA (CUDA) GPU.
import os, sys, platform, shutil, pathlib, sysconfig

IN_COLAB    = "google.colab" in sys.modules   # True when this notebook runs inside Google Colab
MIN_VRAM_GB = 6                             # GPU memory (GB) the training cell of this notebook needs
CAN_TRAIN   = False                           # becomes True below if a suitable CUDA GPU is found
TRAINED     = False                           # becomes True after the training cell finishes successfully
COLAB_URL   = "https://colab.research.google.com/github/arifpucit/Generative-and-Agentic-AI/blob/main/Notebooks/Lec_26_%28FineTuning_Unsloth%29.ipynb"   # this notebook in Google Colab

# Windows fix: `trl` reads its bundled chat-template files with Windows' default encoding (cp1252) and crashes on
# import with a UnicodeDecodeError. Make pathlib read text files as UTF-8 by default for this session.
if sys.platform == "win32" and not sys.flags.utf8_mode:
    _read_text = pathlib.Path.read_text
    pathlib.Path.read_text = lambda self, encoding=None, *args, **kwargs: _read_text(self, encoding or "utf-8", *args, **kwargs)

import torch

print(f"Runtime : {'Google Colab' if IN_COLAB else 'Local machine'} ({platform.system()}, Python {platform.python_version()})")
print(f"PyTorch : {torch.__version__} (CUDA build: {torch.version.cuda or 'none, this is a CPU-only build'})")

if torch.cuda.is_available() and torch.cuda.device_count() > 0:
    major, minor = torch.cuda.get_device_capability(0)
    total_gb     = torch.cuda.get_device_properties(0).total_memory / 1e9
    free_gb      = torch.cuda.mem_get_info()[0] / 1e9
    print(f"GPU     : {torch.cuda.get_device_name(0)} (compute capability {major}.{minor})")
    print(f"VRAM    : {total_gb:.1f} GB total, {free_gb:.1f} GB free right now")
    print(f"bf16    : {'supported' if torch.cuda.is_bf16_supported() else 'not supported (fp16 is used instead)'}")
    if f"sm_{major}{minor}" not in torch.cuda.get_arch_list():
        print(f"\n⚠️  This PyTorch build has no kernels for your GPU (sm_{major}{minor}); supported: {', '.join(torch.cuda.get_arch_list())}")
    elif total_gb < MIN_VRAM_GB:
        print(f"\n⚠️  Training in this notebook needs about {MIN_VRAM_GB} GB of GPU memory, your GPU has {total_gb:.1f} GB.")
    else:
        CAN_TRAIN = True
        if free_gb < MIN_VRAM_GB:
            print(f"\n⚠️  Only {free_gb:.1f} GB is free right now: close other programs that use the GPU, or training may run out of memory.")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    print("GPU     : Apple Silicon (MPS) - fine for inference, but training in this notebook needs an NVIDIA CUDA GPU")
elif shutil.which("nvidia-smi") and torch.version.cuda is None:
    print("GPU     : an NVIDIA GPU is present, but the installed PyTorch is a CPU-only build!")
    print("          Fix: run `uv sync` again from the repository root (it installs the CUDA build), then restart the kernel.")
else:
    print("GPU     : none detected")

# Windows: Unsloth compiles its GPU kernels with Triton, which uses the C compiler (TinyCC) and CUDA files bundled in the
# triton-windows package. Triton only finds them when this notebook runs on the project's own kernel (the repo's .venv).
if CAN_TRAIN and sys.platform == "win32" and not os.path.exists(
        os.path.join(sysconfig.get_path("platlib"), "triton", "runtime", "tcc", "tcc.exe")):
    CAN_TRAIN = False
    print("\n⚠️  Unsloth (Triton) cannot find its bundled compiler from this kernel: select the project kernel")
    print("    'Python (genai-uv)' (the repo's .venv) via Kernel > Change Kernel, then run this cell again.")

print("\n" + "=" * 100)
if CAN_TRAIN:
    print("✅ You CAN run the training cell of this notebook on this machine.")
elif IN_COLAB:
    print("❌ This Colab session has no usable GPU: select Runtime > Change runtime type > T4 GPU, then run this cell again.")
else:
    print("❌ You can NOT run the training cell on this machine: open this notebook in Google Colab (T4 GPU) to train:")
    print(f"   {COLAB_URL}")
    print("   In Colab: Runtime > Change runtime type > T4 GPU, then run cell 1a, cell 1b and the rest of the notebook.")
    print("   All other cells still run here; the training cell and the cells that need its output print a message and skip.")
    print("   Unsloth itself needs an NVIDIA GPU, so on this machine the Unsloth cells fall back to plain transformers + peft.")
print("=" * 100)

## b. Install dependencies (Google Colab only)
- On your own computer this cell does nothing, because `uv sync` has already installed everything into the virtual environment.
- On Colab it installs the same versions that the course's `uv.lock` uses, so both places run identical code.

```
`unsloth`      | Optimized model loading, LoRA injection, and GGUF export (needs an NVIDIA GPU)
`transformers` | Load pretrained models and tokenizers from HuggingFace
`peft`         | Provides `LoraConfig` and `get_peft_model()` for LoRA adapter injection
`datasets`     | Load datasets from HuggingFace Hub
`accelerate`   | Enables efficient GPU training
`trl`          | Training utilities (`SFTTrainer`, `SFTConfig`)
`python-dotenv`| Loads API keys / tokens from a `.env` file
```

In [ ]:
# ── Run this cell ONLY on Google Colab (it skips itself on your own computer) ────────────────────
import sys, subprocess

if "google.colab" in sys.modules:
    import importlib.metadata, torch
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "unsloth==2026.10.9",
        "unsloth_zoo==2026.10.9",
        "transformers==5.7.0",
        "peft==0.19.1",
        "torchao==0.18.0",      # Colab ships torchao 0.10, which makes peft fail with "incompatible version of torchao"
        "trl==1.3.0",
        "accelerate==1.13.0",
        "datasets==4.8.5",
        "python-dotenv",
    ], check=True)
    if importlib.metadata.version("torch").split("+")[0] != torch.__version__.split("+")[0]:
        print("⚠️  pip upgraded PyTorch: select Runtime > Restart session, then run the notebook again from the top.")
    print("✅ All packages installed successfully!")
else:
    print("Local machine: nothing to install, the uv virtual environment already has every package.")

# <span style='background :lightgreen' >2. Dataset Curation</span>
- The dataset used for fine-tuning is a **custom synthetic Q&A dataset** built around **Dr. Muhammad Arif Butt** — covering his academic background, teaching experience, research interests, contact information and YouTube channel — curated specifically to teach the model to answer questions about him accurately and naturally.
## How the Dataset Was Curated
The dataset was created following a six-step curation pipeline — the same pipeline students can follow to build their own domain-specific datasets:
- **Step 1 — Collection:** Raw facts were gathered from Dr. Arif Butt's official website (https://arifbutt.me) and YouTube channel — covering all relevant topics including academic position, qualifications, research interests, courses taught and contact details
- **Step 2 — AI-Assisted Q&A Generation:** Each collected fact was fed into an AI model with a prompt instructing it to generate **3 to 5 different question variations per fact** — this rephrasing improves model generalisation by teaching it to answer the same fact regardless of how the question is phrased:
```
Fact: "Dr. Arif Butt is an Assistant Professor at PUCIT"

Generated question variations:
  "Who is Dr. Muhammad Arif Butt?"
  "Tell me about Dr. Arif Butt"
  "What is Arif Butt's current position?"
  "Where does Dr. Muhammad Arif Butt work?"
  "Which university does Dr. Arif Butt teach at?"
```
- **Step 3 — Cleaning and Validation:** All generated Q&A pairs were reviewed to remove duplicates, fix factual inaccuracies, ensure consistent tone and style, and discard any incomplete or low quality examples — human review of every example is critical at this stage since AI-generated data can contain subtle errors
- **Step 4 — Formatting:** Each Q&A pair was structured into the standard **chat `messages` format** (a list of `{role, content}` dicts). Later in this notebook every example is converted into TinyLlama-Chat's prompt template — this is not optional — the chat model was fine-tuned on that exact template and responds best when the fine-tuning data matches it:
```json
{
  "messages": [
    {"role": "user",      "content": "Who is Dr. Muhammad Arif Butt?"},
    {"role": "assistant", "content": "Dr. Muhammad Arif Butt is an Assistant Professor
                                      at the Department of Data Science,
                                      University of the Punjab..."}
  ]
}
```
- **Step 5 — Balancing:** Questions were distributed evenly across all topics — academic background, teaching experience, research interests, contact information and YouTube channel — preventing the model from becoming biased toward one topic while forgetting others
- **Step 6 — Upload to HuggingFace Hub:** The final cleaned and formatted dataset was saved as `.jsonl` and uploaded as a public dataset so it can be loaded in one line anywhere:

#### 🔗 **Dataset:** [arif-butt/arifbutt_dataset](https://huggingface.co/datasets/arif-butt/arifbutt_dataset)


## Students should build their own synthetic Dataset
- Pick any domain, a product, a company or a public figure.
- Collect facts from public sources;  websites, Wikipedia, official pages.
- Feed the facts into an AI model and ask it to generate 3 to 5 question variations per fact.
- Review and clean all generated pairs, remove duplicates and fix any inaccuracies.
- Format each pair as chat `messages` (role/content) and save as `.jsonl`.
- Upload to HuggingFace Hub using `dataset.push_to_hub()` and your dataset is ready to use.

# <span style='background :lightgreen' >3. Configuration of Model, Dataset, LoRA Adapters, and Training  Hyperparameters</span>

- The code cell below defines the base model, dataset source, output directories, LoRA configuration, and training hyperparameters required for fine-tuning.
- It centralizes all experiment settings such as batch size, learning rate, and sequence length to make the training pipeline easy to configure and reproduce.
- Finally, it creates the required output directories for saving LoRA adapters and the merged final model.

In [ ]:
import torch                 # PyTorch library for tensor operations and GPU acceleration
import gc                    # garbage collector for freeing up memory during training
if CAN_TRAIN:                            # Unsloth needs an NVIDIA GPU (import it before trl / transformers so it can patch them)
    from unsloth import FastLanguageModel    # Unsloth's optimized model loading with faster training kernels
from datasets import load_dataset        # HuggingFace dataset loading utility
from trl import SFTTrainer, SFTConfig    # TRL's supervised fine-tuning trainer and config

BASE_MODEL_ID   = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"    # base model identifier from HuggingFace Hub
HF_DATASET_NAME = "arif-butt/arifbutt_dataset"            # dataset identifier for fine-tuning
ADAPTER_OUTPUT_DIR      = "./tinyllama-unsloth-lora-adapter"      # directory to save LoRA adapter weights
MERGED_OUTPUT_DIR      = "./tinyllama-unsloth-merged"                    # directory to save merged model


SYSTEM_PROMPT = (                                  # system instruction prepended to every conversation
    "You are a helpful assistant."                 # sets model behavior to be helpful and assistant-like
)

# LoRA Hyperparameters
LORA_R       = 16                                 # rank of LoRA adapter matrices — controls trainable parameter count
LORA_ALPHA   = 32                                 # scaling factor for LoRA updates — alpha/r = 2.0
LORA_DROPOUT = 0.05                               # dropout probability for LoRA layers — prevents overfitting

# Training Hyperparameters
NUM_EPOCHS     = 3                               # number of complete passes through training dataset
BATCH_SIZE     = 1                               # samples per GPU per forward/backward pass
GRAD_ACCUM     = 2                               # batches to accumulate before weight update
LEARNING_RATE  = 2e-4                            # optimizer step size for parameter updates
MAX_SEQ_LENGTH = 256                             # maximum tokens per training sample
WARMUP_steps  = 10                               # initial steps where learning rate increases gradually
SAVE_STEPS     = 50                              # steps between saving model checkpoints
LOGGING_STEPS  = 10                              # steps between logging training metrics
SEED           = 42                              # fixed random seed for reproducibility

# ── Speed settings, picked automatically for your GPU ──────────
# With batch 1 the GPU sits mostly idle (~30% busy, 2.4 GB used). Ampere or newer GPUs (RTX 30xx/40xx/50xx, A100, ...)
# get a batch of 8 → training is about 7x faster (RTX 5070 Ti: ~2 min instead of 15 min).
# Older GPUs such as Colab's T4 keep the original batch 1 x 2.
FAST_GPU = CAN_TRAIN and torch.cuda.get_device_capability(0)[0] >= 8
if FAST_GPU:
    BATCH_SIZE = 8                               # 8 samples per step (peak ≈ 5.6 GB)
    GRAD_ACCUM = 1                               # effective batch size = 8 * 1 = 8
# Unsloth's gradient checkpointing saves memory by recomputing activations, but makes training ~1.5x slower:
# only use it on GPUs with less than 12 GB of memory.
GRAD_CHECKPOINTING = "unsloth" if CAN_TRAIN and torch.cuda.get_device_properties(0).total_memory / 1e9 < 12 else False
if CAN_TRAIN:
    print(f"Speed settings for {torch.cuda.get_device_name(0)}: batch {BATCH_SIZE} x {GRAD_ACCUM}, "
          f"gradient checkpointing {'on (unsloth)' if GRAD_CHECKPOINTING else 'off'}")

# <span style='background :lightgreen' >4. Load Base Model + Tokenizer</span>

**Unsloth replaces `transformers` here.**

`FastLanguageModel.from_pretrained()` loads the model with:
- Unsloth's optimized CUDA kernels — up to 2–5× faster training than the standard transformers + peft stack
- Automatic precision: bf16 on GPUs that support it (Ampere / RTX 30xx and newer), fp16 on older GPUs such as the T4
- `max_seq_length` set at load time — no truncation surprises during training

> On a machine without an NVIDIA GPU this cell loads **only the tokenizer** (needed to prepare the dataset below); the Unsloth model is skipped.

In [ ]:
if CAN_TRAIN:
    print("Loading TinyLlama with Unsloth...")

    # FastLanguageModel.from_pretrained loads both model & tokenizer together
    # Unsloth's implementation provides 2-5x faster training and 70% less memory usage
    # Optimized specifically for LoRA fine-tuning with fused kernels
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL_ID,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,           # let Unsloth auto-select: bf16 on GPUs that support it, otherwise fp16
        load_in_4bit=False,    # True = QLoRA (4-bit quantized base model); False = standard 16-bit LoRA
    )

    print("Model + Tokenizer ready!")
    print(f"Max sequence length : {MAX_SEQ_LENGTH}")
    print(f"Model dtype         : {next(model.parameters()).dtype}")
else:
    from transformers import AutoTokenizer
    model = None
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
    print("⏭️  No NVIDIA GPU: the Unsloth model is not loaded, only the tokenizer (used to prepare the dataset below).")

# Pad with </s> on the right, exactly as in Lec-24/25. Unsloth's tokenizer pads with <unk> on the LEFT: because our labels are
# a copy of input_ids, the ~170 <unk> padding tokens of every example (which the model cannot even see, attention_mask = 0)
# would be counted in the loss, which then stays stuck at ≈ 7 for the whole training and the model learns much worse.
tokenizer.pad_token    = tokenizer.eos_token      # TinyLlama has no dedicated pad token, so EOS is reused
tokenizer.padding_side = "right"                  # real tokens first, padding after them

# <span style='background :lightgreen' >5. Prepare Dataset</span>

In [4]:
from datasets import load_dataset
from dotenv import load_dotenv      # Imports the load_dotenv function, which loads environment variables from a .env file into your Python environment.
from huggingface_hub import login   # Imports the login() function that allows you to authenticate with the Hugging Face Hub using an access token.
import os                           # Imports Python’s built-in os module so you can access environment variables (like your HF token).

# The following login() call is only necessary if you want to access private repos, or to use private models or to push models/datasets
'''
load_dotenv('../keys/.env', override=True)
hf_token = os.getenv('HF_TOKEN')
login(token=hf_token)
'''

raw_dataset = load_dataset("arif-butt/arifbutt_dataset")
print("✓ Dataset loaded successfully!")
raw_dataset

drarif.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/1926 [00:00<?, ? examples/s]

✓ Dataset loaded successfully!


DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 1926
    })
})

In [5]:
print("\nSample examples from the training split:\n")
for idx in range(5):
    sample = raw_dataset["train"][idx]
    print(f"Example {idx + 1}: {sample}")


Sample examples from the training split:

Example 1: {'messages': [{'role': 'user', 'content': 'Who is Dr. Muhammad Arif Butt?'}, {'role': 'assistant', 'content': 'Dr. Muhammad Arif Butt is an accomplished Assistant Professor at the Department of Data Science, University of the Punjab (PU), Lahore, Pakistan. He has over 33 years of experience in teaching and management, with expertise in cybersecurity, operating systems, and artificial intelligence.'}]}
Example 2: {'messages': [{'role': 'user', 'content': 'Tell me about Dr. Arif Butt'}, {'role': 'assistant', 'content': "Dr. Muhammad Arif Butt is an Assistant Professor at Punjab University's Data Science department. He's a technology entrepreneur and cybersecurity expert with 33+ years of experience, combining academic excellence with industry innovation."}]}
Example 3: {'messages': [{'role': 'user', 'content': "What is Arif butt's current position?"}, {'role': 'assistant', 'content': 'Dr. Muhammad Arif Butt currently serves as an Assi

## Format the [arif-butt/arifbutt_dataset](https://huggingface.co/datasets/arif-butt/arifbutt_dataset)
- The dataset uses a `messages` format — a list of `{role, content}` dicts. We must convert each example into the exact chat-template string TinyLlama-Chat expects (a Zephyr-style template with `<|system|>`, `<|user|>`, `<|assistant|>` tags and `</s>` after every turn — note that this is *not* ChatML, which uses `<|im_start|>` / `<|im_end|>`):

```
<|system|>
You are a helpful assistant.</s>
<|user|>
Who is Dr. Arif Butt?</s>
<|assistant|>
Dr. Muhammad Arif Butt is an Assistant Professor...</s>
```

**Steps inside `format_example()`:**
1. Reads the `messages` list (user / assistant turns) of each example
2. Builds the TinyLlama chat-template string for each example
3. `dataset.map()` applies this to every example in the dataset

**Tokenization:**
- `labels = input_ids` — for causal LM, the model learns to predict the next token, so labels are the same as inputs
- `truncation=True` — examples longer than `MAX_SEQ_LENGTH` are cut off
- `padding="max_length"` — shorter examples are padded to the same length

In [ ]:
################# CONVERT DATASET ROWS INTO TINYLLAMA CHAT FORMAT ###############################

# ── Format Function ───────────────────────────────────────────────────────────
def format_example(example):
    """Format messages into TinyLlama chat-template conversation."""
    text = "<|system|>\nYou are a helpful assistant.</s>\n"  # start every conversation with a system prompt that sets the assistant's behaviour and persona
    for msg in example["messages"]:         # iterate over each message in the conversation — each message is a dict with "role" and "content" keys
        role    = msg["role"]               # extract the role of the message sender — either "user" or "assistant"
        content = msg["content"].strip()    # extract and clean the message text — strip() removes any leading or trailing whitespace
        if role == "user":
            text += f"<|user|>\n{content}</s>\n"        # wrap user message in TinyLlama's chat-template user tags — <|user|> opens and </s> closes the turn
        elif role == "assistant":
            text += f"<|assistant|>\n{content}</s>\n"   # wrap assistant message in TinyLlama's chat-template assistant tags — model learns to generate text between these tags
    return {"formatted_text": text}   # return the fully formatted conversation as a new column called "formatted_text" that replaces the original "messages" column

# ── Tokenize Function ─────────────────────────────────────────────────────────
def tokenize(example):
    """Convert formatted text into token IDs."""
    result = tokenizer(
                    example["formatted_text"],      # pass the chat-template formatted conversation string to the tokenizer for conversion to token IDs
                    truncation=True,                # if the formatted text exceeds MAX_SEQ_LENGTH tokens — cut it off and keep only the first MAX_SEQ_LENGTH tokens
                    max_length=MAX_SEQ_LENGTH,      # set the fixed sequence length to MAX_SEQ_LENGTH — all sequences will be either truncated or padded to exactly this length
                    padding="max_length",           # if the formatted text is shorter than MAX_SEQ_LENGTH — pad it on the right with the pad token (EOS token) to reach exactly MAX_SEQ_LENGTH tokens
                )
    result["labels"] = result["input_ids"].copy()   # for causal language model training the labels are identical to input_ids — the model learns to predict the next token by shifting input one position right internally
    return result   # return tokenized output containing input_ids, attention_mask and labels — all of length MAX_SEQ_LENGTH

# ── Apply Formatting ──────────────────────────────────────────────────────────
formatted_dataset = raw_dataset.map(
                        format_example,                                    # apply the format_example function to every row in the dataset
                        remove_columns=raw_dataset["train"].column_names,  # drop all original columns (messages, etc.) and keep only the new "formatted_text" column
                        )

# ── Tokenize Dataset ──────────────────────────────────────────────────────────
tokenized_dataset = formatted_dataset.map(
                        tokenize,                          # apply the tokenize function to every row in the formatted dataset
                        batched=True,                      # process multiple examples at once for significantly faster tokenization
                        remove_columns=["formatted_text"], # drop the "formatted_text" string column — no longer needed after tokenization
                        )

# ── Train / Eval Split ────────────────────────────────────────────────────────
spl = tokenized_dataset["train"].train_test_split(test_size=0.1, seed=SEED)  # randomly split the tokenized training data into 90% train and 10% evaluation using fixed seed for reproducibility

train_dataset = spl["train"]  # assign the 90% split to train_dataset — used to update LoRA adapter weights during training
eval_dataset  = spl["test"]   # assign the 10% split to eval_dataset  — used to monitor loss on unseen examples during training without updating any weights

print("Dataset prepared")
print(f"Train examples:\n {train_dataset}")  # print the train dataset object showing number of examples, column names and data types
print(f"Eval examples:\n {eval_dataset}")    # print the eval  dataset object showing number of examples, column names and data types

In [7]:
# Print fully tokenized training example that will be fed to the model during fine-tuning of TinyLlama-1.1B-Chat-v1.0.
print(train_dataset[0])

{'input_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 529, 29989, 5205, 29989, 29958, 13, 3492, 526, 263, 8444, 20255, 29889, 2, 13, 29966, 29989, 1792, 29989, 29958, 13, 25125, 4942, 29889, 826, 361, 505, 263, 6189, 29257, 3236, 373, 670, 14711, 8242, 29973, 2, 13, 29966, 29989, 465, 22137, 29989, 29958, 13, 8241, 29892, 4942, 29889, 29610, 826, 361, 1205, 29873, 29915, 29879, 14711, 8242, 525, 29931, 799, 29876, 2973, 826, 361, 29915, 756, 263, 6189, 29257, 3236, 411, 29871, 29945, 19707, 3625, 363, 8041

> **input_ids:** 1 → start-of-sequence token, 13 → newline token, 2 → end-of-sequence token `</s>`, which is also our padding token. We pad on the **right**, so the padding comes *after* the real text, up to `MAX_SEQ_LENGTH = 256` tokens (we used `padding = "max_length"`). (Unsloth's own tokenizer would pad with `<unk>` (id 0) on the left: with `labels = input_ids` those padding tokens would be counted in the loss, so we changed it in section 4.)

> **attention_mask:** This tensor tells the transformer which tokens to pay attention to. 1 →  real conversation tokens, 0 →  padding and transformer ignores tokens where mask = 0

> **labels:** For causal language modeling, the labels are simply input_ids. During training the model learns: `Predict token_(t+1) from token_1 ... token_t`

In [8]:
sample = train_dataset[0]
print("Decoded text:\n")
print(tokenizer.decode(sample["input_ids"]))

Decoded text:

<unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><unk><s> <|system|>
You are a helpful assistant.</s>
<|user|>
Does Dr. Arif have a Machine Learning course on his Y

# <span style='background :lightgreen' >6. Configure and Inject LoRA Adapters</span>
**Unsloth replaces `peft` here.**

`FastLanguageModel.get_peft_model()` injects optimized LoRA adapter matrices directly into the model using Unsloth's faster backward pass kernels — replacing the standard PEFT approach.

**`target_modules`** — which layers to adapt:

| Module | Layer Type |
|---|---|
| `q_proj`, `k_proj`, `v_proj`, `o_proj` | Attention — Query, Key, Value, Output |
| `gate_proj`, `up_proj`, `down_proj` | MLP Feed-Forward layers |

**`use_gradient_checkpointing = "unsloth"`** — Unsloth's custom gradient checkpointing saves additional VRAM beyond standard PyTorch checkpointing, but recomputing activations makes training ~1.5× slower. This notebook therefore turns it on only on GPUs with less than 12 GB (`GRAD_CHECKPOINTING` in section 3) and uses a batch of 8 on Ampere or newer GPUs (`FAST_GPU`).

In [ ]:
if CAN_TRAIN:
    print("Injecting LoRA adapters with Unsloth...")

    # Using Unsloth's optimized get_peft_model instead of standard PEFT's get_peft_model
    # WHY? Standard PEFT's get_peft_model would use generic LoRA implementation with slower kernels
    # Unsloth's version uses fused kernels and memory optimizations that are 2-5x faster
    # Standard approach: model = get_peft_model(model, lora_config)  # slower, generic implementation
    # Unsloth approach: direct method call with optimized parameters
    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=0.0,          # 0 is fastest: Unsloth's fused kernels are optimized for dropout = 0 (non-zero works but is slower)
        bias="none",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        use_gradient_checkpointing=GRAD_CHECKPOINTING,  # "unsloth" (Unsloth-specific checkpointing) on GPUs < 12 GB, else off for speed
        random_state=SEED,
    )

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"Trainable parameters : {trainable:,}")
    print(f"Total parameters     : {total:,}")
    print(f"Trainable %          : {100 * trainable / total:.2f}%")
else:
    print('⏭️  Skipping LoRA injection: Unsloth needs an NVIDIA GPU.')

# <span style='background :lightgreen' >7. Configure Training</span>
- The following code cell defines the entire training behavior for the fine-tuning process:
    - training duration
    - batch sizes
    - optimizer
    - learning rate schedule
    - checkpointing
    - evaluation
    - reproducibility
- It acts as the central configuration object for the training loop used by the Hugging Face Trainer

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"          # disables tokenizer parallelism to avoid deadlocks during training

training_args = SFTConfig(
    output_dir=ADAPTER_OUTPUT_DIR,                              # directory where training checkpoints and logs are saved
    num_train_epochs=NUM_EPOCHS,                        # number of complete passes through the entire training dataset
    per_device_train_batch_size=BATCH_SIZE,             # number of training samples processed per GPU in each step
    gradient_accumulation_steps=GRAD_ACCUM,             # number of steps to accumulate gradients before weight update
    learning_rate=LEARNING_RATE,                        # step size for optimizer updates during training
    warmup_steps=WARMUP_steps,                          # number of initial steps where learning rate increases gradually
    lr_scheduler_type="cosine",                         # learning rate schedule that decreases LR using cosine decay
    dataloader_num_workers=0,                           # number of subprocesses for data loading (0 = main process)
    fp16=CAN_TRAIN and not torch.cuda.is_bf16_supported(),  # fp16 mixed precision on GPUs without bf16 support (e.g. T4)
    bf16=CAN_TRAIN and torch.cuda.is_bf16_supported(),      # bf16 mixed precision on newer GPUs (Ampere / RTX 30xx and newer)
    logging_steps=LOGGING_STEPS,                        # number of steps between logging training metrics
    eval_strategy="steps",                              # specifies that evaluation occurs periodically during training
    eval_steps=50,                                      # number of steps between evaluation runs on validation dataset
    save_strategy="steps",                              # specifies that checkpoints are saved periodically during training
    save_steps=50,                                      # number of training steps between saving model checkpoints
    load_best_model_at_end=True,                        # automatically reload the checkpoint with the best evaluation metric
    metric_for_best_model="eval_loss",                  # metric used to determine which checkpoint is the best model
    weight_decay=0.01,                                  # regularization parameter that penalizes large weights
    save_total_limit=2,                                 # maximum number of checkpoints to keep (older ones are deleted)
    max_length=MAX_SEQ_LENGTH,                          # maximum number of tokens allowed per training sample
    dataset_text_field="text",                          # only used for raw-text datasets; ours is already tokenized, so it is ignored
    report_to="none",                                   # disables external experiment tracking tools (e.g., WandB)
    seed=SEED,                                          # fixed random seed to ensure reproducible training results
    dataset_num_proc=1,                                 # number of processes for dataset preprocessing
)

print("Training config ready!")                         # confirmation that training configuration is complete
print(f"Epochs               : {NUM_EPOCHS}")           # display number of training epochs configured
print(f"Effective batch size : {BATCH_SIZE * GRAD_ACCUM}")  # display actual batch size after gradient accumulation
print(f"Learning rate        : {LEARNING_RATE}")        # display the learning rate being used for training

# <span style='background :lightgreen' >8. Train the Model (needs an NVIDIA GPU)</span>
**What happens inside `trainer.train()`:**
```
For each batch:
  1. Forward pass   — feed tokens through model, get predictions
  2. Loss           — measure how wrong predictions are (cross-entropy)
  3. Backprop       — calculate gradients
  4. Update         — adjust only LoRA adapter weights
  5. Repeat
```

**How to read the training table:**
```
`Step`            | How many batches have been processed
`Training Loss`   | Loss on training data — should go down
`Validation Loss` | Loss on unseen data — should follow training loss
```
**Diagnosing your training:**
-  Both losses going down together → healthy training
-  Train loss low, eval loss rising → overfitting → reduce epochs
-  Both losses staying high → underfitting → increase epochs or LORA_R

**This cell runs only if the runtime check in section 1 found a suitable NVIDIA (CUDA) GPU** — otherwise it prints a message and is skipped.

| Hardware | Time to run this cell |
|---|---|
| Google Colab, Tesla T4 (16 GB) | ≈ 30 min (measured with gradient checkpointing on; now shorter) |
| Your PC: NVIDIA GeForce RTX 5070 Ti (16 GB) | 2.6 min (batch 8: 651 steps, peak GPU memory 5.3 GB) |

You can see the total steps, train loss, training time and peak GPU memory in the output of this cell.

In [ ]:
if CAN_TRAIN:
    # Using SFTTrainer (Supervised Fine-Tuning Trainer) from TRL library
    # WHY SFTTrainer with Unsloth?
    # - SFTTrainer is specifically designed for instruction fine-tuning of LLMs
    # - It automatically handles:
    #   1. Formatting prompts with chat templates
    #   2. Masking loss on prompt tokens (only compute loss on responses)
    #   3. Packing sequences for efficiency
    #   4. Managing padding and attention masks
    # - Unsloth integrates seamlessly with SFTTrainer - no special modifications needed
    # - Standard alternative would be Trainer from transformers, but SFTTrainer provides:
    #   * Built-in support for instruction datasets
    #   * Automatic prompt formatting
    #   * Better handling of conversational data
    trainer = SFTTrainer(
        model=model,                    # Unsloth model with LoRA adapters already injected
        processing_class=tokenizer,     # tokenizer for encoding text (SFTTrainer expects processing_class not tokenizer in newer versions)
        train_dataset=train_dataset,    # tokenized training split
        eval_dataset=eval_dataset,
        args=training_args
    )

    import time
    print("Training started...")
    start_time = time.perf_counter()
    train_result = trainer.train()      # Unsloth's optimizations work automatically through the model
    print("Training complete!")
    print(f"Train loss      : {train_result.training_loss:.4f}")
    print(f"Total steps     : {train_result.global_step}")
    TRAINED = True
    train_minutes = (time.perf_counter() - start_time) / 60
    print(f"Training time   : {train_minutes:.1f} min on {torch.cuda.get_device_name(0)}")
    print(f"Peak GPU memory : {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
else:
    print('⏭️  Skipping training: no suitable NVIDIA (CUDA) GPU on this machine (see the runtime check in section 1).')
    print('    To train, open this notebook in Google Colab with a T4 GPU. The remaining cells still run and use the')
    print('    fine-tuned model that is already published on the Hugging Face Hub.')

# <span style='background :lightgreen' >9. Training Loss Plot</span>
This cell plots the training and evaluation loss as a line chart so you can visually diagnose training health.

- 🔵 **Blue line** — Training loss
- 🔴 **Red line** — Validation loss

Both lines going down smoothly and staying close together = healthy training. The plot is also saved as `loss_curve.png`.

In [ ]:
if TRAINED:
    import matplotlib.pyplot as plt

    logs = trainer.state.log_history

    train_loss = [(x['step'], x['loss'])      for x in logs if 'loss' in x and 'eval_loss' not in x]
    eval_loss  = [(x['step'], x['eval_loss']) for x in logs if 'eval_loss' in x]

    plt.figure(figsize=(10, 5))
    plt.plot(*zip(*train_loss), label='Training Loss',   color='royalblue', marker='o', markersize=3)
    plt.plot(*zip(*eval_loss),  label='Validation Loss', color='tomato',    marker='s', markersize=4)
    plt.xlabel('Steps')
    plt.ylabel('Loss')
    plt.title('Training vs Validation Loss — TinyLlama Finetuning')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig('loss_curve.png', dpi=150)
    plt.show()
    print("Graph saved → loss_curve.png")
else:
    print('⏭️  No training was run in this session, so there is no loss curve to plot.')

# <span style='background :lightgreen' >10. Save the Adapter Weights, Push to HF and Do Inference</span>

## a. Save the Adapter Weights
- After training we save **only the LoRA adapter weights** (48.16 MB), not the full merged model weights which will be greater than 2.2 GB.
- The saved folder contains:
    - `adapter_config.json`       — LoRA architecture definition
    - `adapter_model.safetensors` — the actual trained weights, around 48 MB only
    - `tokenizer files`           — if tokenizer saved alongside adapters
- Can be pushed on hugging face hub
- The advantages of this option are:
    -  The size of files to be saved is small
    -  If pushed on HF hub, any one having TinyLlama base can download the adapter weights and use them with base model

####  After `model.save_pretrained()` and `tokenizer.save_pretrained()`, the ADAPTER_OUTPUT_DIR (`./tinyllama-unsloth-lora-adapter`) contains:

| File | Size | Contents |
|---|---|---|
| `adapter_model.safetensors` | 48.16 MB | The **LoRA adapter weights only** — the small delta matrices (A & B) injected into the base model layers. Does **not** contain base model weights. |
| `adapter_config.json` | 737 B | LoRA configuration: rank (`r`), alpha, dropout, target modules, task type. Needed to reconstruct the PEFT model. |
| `tokenizer.json` | 1.76 MB | Fast tokenizer data — full vocabulary, BPE merge rules, and special token mappings. |
| `tokenizer_config.json` | 1.3 KB | Tokenizer metadata — class name, special tokens (`bos_token`, `eos_token`, `pad_token`), padding side, etc. |
| `tokenizer.model` | 488.0 KB | The raw SentencePiece model file used by the slow tokenizer for subword tokenization. |
| `special_tokens_map.json` | 437 B | Maps special token names (`bos_token`, `eos_token`, `unk_token`) to their string values. |
| `checkpoint-350/` | 4.0 KB | Mid-training checkpoint saved automatically at step 350. |
| `checkpoint-400/` | 4.0 KB | Mid-training checkpoint saved automatically at step 400 (latest). |
| `README.md` | 5.0 KB | Auto-generated model card describing the adapter. |



>- **Limitation of this option is, the uploaded adapter weights cannot be used as a standalone model, for which you have to merge**

In [ ]:
if TRAINED:
    # Save LoRA Adapter Weights Only (Unsloth)

    # Save adapter weights and tokenizer after training
    print("Saving LoRA adapter weights...")
    model.save_pretrained(ADAPTER_OUTPUT_DIR)               # saves adapter_config.json and adapter_model.safetensors
    tokenizer.save_pretrained(ADAPTER_OUTPUT_DIR)           # saves tokenizer files for reuse

    print(f"Adapter saved to: {ADAPTER_OUTPUT_DIR}")
    print(f"Size: ~48 MB (adapter weights only)")
else:
    print('⏭️  Skipping save: no adapter was trained in this session (the inference cells below use the adapter published on the Hub).')

In [ ]:
import os

if os.path.exists(ADAPTER_OUTPUT_DIR):
    print(f"Files saved in: {ADAPTER_OUTPUT_DIR}\n")
    for fname in sorted(os.listdir(ADAPTER_OUTPUT_DIR)):
        fpath = os.path.join(ADAPTER_OUTPUT_DIR, fname)
        size  = os.path.getsize(fpath)
        if size >= 1_048_576:
            size_str = f"{size / 1_048_576:.2f} MB"
        elif size >= 1024:
            size_str = f"{size / 1024:.1f} KB"
        else:
            size_str = f"{size} B"
        print(f"  {fname:<40} {size_str}")
else:
    print(f"Directory '{ADAPTER_OUTPUT_DIR}' not found.")
    print("Run the training and save steps first before listing files.")

## b. Push the Files to HuggingFace Hub
Push the LoRA adapter files to the HuggingFace Hub, so anyone who has the TinyLlama base model can download and use your adapter.
- The repo is created under **your own** Hugging Face username (read from your token), e.g. `your-username/<repo-name>`.

**Before pushing, you need a Write token:**  
Go to → **huggingface.co → Profile → Settings → Access Tokens → New Token → Type: Write**

In [ ]:
import os

if os.path.exists(os.path.join(ADAPTER_OUTPUT_DIR, "adapter_model.safetensors")):
    from huggingface_hub import HfApi, create_repo, login
    from getpass import getpass

    # 1. LOGIN (SECURE INPUT) — needs a token of type "Write"
    hf_token = getpass("Enter your Hugging Face API token: ")
    login(token=hf_token)

    # 2. REPO DETAILS — the repo is created under YOUR Hugging Face username
    api = HfApi()
    repo_id = f"{api.whoami()['name']}/tinyllama-unsloth-lora-adapter"

    # 3. CREATE REPO (IF NOT EXISTS)
    print("Creating repo (if needed)...")
    create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)
    print(f"Repo ready: https://huggingface.co/{repo_id}")

    # 4. UPLOAD FILES
    print("Uploading adapter files (this may take a while)...")
    api.upload_folder(folder_path=ADAPTER_OUTPUT_DIR, repo_id=repo_id, repo_type="model")

    print("Upload complete!")
    print(f"\nYour adapter files is live at: https://huggingface.co/{repo_id}")
else:
    print('⏭️  Skipping upload: no local adapter files found (they are created by the cells above, after training).')

## c. Inference with loading LoRa Adapter from hugging face

In [ ]:
# ── Inference with Unsloth + LoRA Adapter from Hub ────────────────────────────

import torch
from peft import PeftModel
from transformers import pipeline

# ── Model Configuration
BASE_MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"              # base model identifier
ADAPTER_REPO = "arif-butt/tinyllama-unsloth-lora-adapter"         # adapter repo on HF Hub (replace with yours after pushing)

if CAN_TRAIN:
    # ── Load Base Model with Unsloth Optimizations
    print("Loading base model with Unsloth...")
    base_model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL_ID,                                 # base model to load
        max_seq_length=512,                                       # maximum sequence length
        dtype=torch.float16,                                      # use float16 for efficiency
        load_in_4bit=False,                                       # disable 4-bit quantization
    )
else:
    # ── No NVIDIA GPU: Unsloth is unavailable, so load the same base model with plain transformers
    from transformers import AutoModelForCausalLM, AutoTokenizer
    print("Loading base model with transformers (Unsloth needs an NVIDIA GPU)...")
    base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, dtype=torch.float16, device_map="auto")
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
print("Base model loaded")

# ── Load and Attach LoRA Adapters from Hub
print("Loading LoRA adapters from HuggingFace Hub...")
model = PeftModel.from_pretrained(
    base_model,                                                   # base model to attach the adapters to
    ADAPTER_REPO,                                                 # HuggingFace repo with adapter weights
)
print("Adapters attached")

# ── Create Text Generation Pipeline
print("Creating pipeline...")
pipe = pipeline(
    "text-generation",                                            # text generation task
    model=model,                                                  # fine-tuned model (already placed on the GPU/CPU)
    tokenizer=tokenizer,                                          # matching tokenizer
)

In [ ]:
# Single Prompt Test
prompt = "Q: Name all the courses Arif butt teach?\nA:"          # test prompt

print("\n" + "="*50)
print(f"Prompt: {prompt}")

output = pipe(
    prompt,                                                       # input prompt
    max_new_tokens=100,                                           # maximum new tokens to generate
    temperature=0.2,                                              # low temp for factual answers
    do_sample=True,                                               # enable sampling
    pad_token_id=tokenizer.eos_token_id,                          # prevent infinite generation
)

print(f"Response: {output[0]['generated_text']}")
print("="*50)

### Before merging, we free up VRAM by deleting the trainer and model from memory. This prevents Out-of-Memory errors when reloading the base model for merging.

In [ ]:
import gc
import torch

for name in ("trainer", "model"):      # these only exist if the earlier cells created them (e.g. training was run)
    globals().pop(name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("GPU memory cleared!")
if torch.cuda.is_available():
    print(f"Allocated : {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"Reserved  : {torch.cuda.memory_reserved() / 1e9:.2f} GB")

# <span style='background :lightgreen' >11. Merge Adapter Weights, Push to HF and Do Inference</span>

## a. Save the Complete Merged  Model
- Now that we have saved the adapter weights, we can now create the final standalone model by merging the LoRA adapters into the base model.
- Before merge → need both base model + adapter files to run
- After merge → one single file, no PEFT library needed at inference time
- **What Gets Saved After Both Steps:**
```
./tinyllama-unsloth-lora-adapter/              ← ADAPTER_OUTPUT_DIR
  ├── adapter_config.json               ← LoRA config (r=16, alpha=32, target modules)
  ├── adapter_model.safetensors         ← LoRA adapter weights only (~48 MB)
  ├── tokenizer.json                    ← full tokenizer
  ├── tokenizer_config.json             ← tokenizer settings
  ├── special_tokens_map.json           ← special tokens
  └── tokenizer.model                   ← tokenizer binary

./tinyllama-unsloth-merged/                     ← MERGED_OUTPUT_DIR
  ├── config.json                       ← full model architecture config
  ├── generation_config.json            ← default generation settings
  ├── model.safetensors                 ← FULL merged weights (~2.2 GB)
  ├── tokenizer.json                    ← full tokenizer
  ├── tokenizer_config.json             ← tokenizer settings
  ├── special_tokens_map.json           ← special tokens
  └── tokenizer.model                   ← tokenizer binary
```

In [ ]:
import os
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

# Use the adapter you trained above; if you skipped training (no GPU), use the adapter published on the Hub instead
ADAPTER_SOURCE = ADAPTER_OUTPUT_DIR if os.path.exists(os.path.join(ADAPTER_OUTPUT_DIR, "adapter_config.json")) else "arif-butt/tinyllama-unsloth-lora-adapter"
print(f"Adapter source: {ADAPTER_SOURCE}")

if CAN_TRAIN:
    # ── Load Base Model with Unsloth Optimizations
    print("Loading base model with Unsloth...")
    base_model, _ = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL_ID,                                 # base model to load
        max_seq_length=512,                                       # maximum sequence length
        dtype=torch.float16,                                      # use float16 for efficiency
        load_in_4bit=False,                                       # disable 4-bit quantization
    )
else:
    # ── No NVIDIA GPU: load the same base model with plain transformers
    print("Loading base model with transformers (Unsloth needs an NVIDIA GPU)...")
    base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, dtype=torch.float16, device_map="auto")
print("Base model loaded")

# Putting a trained overlay on top of the original model. It reads the adapter architecture from `adapter_config.json`, loads the trained weights, and clips them onto the exact layers they were trained on.
print("Attaching saved LoRA adapters...")
finetuned_model = PeftModel.from_pretrained(base_model, ADAPTER_SOURCE)

#  During training, adapters lived separately alongside the frozen weights. Merging permanently bakes them in
print("Merging adapter weights into base model...")
finetuned_model = finetuned_model.merge_and_unload()     # Final Weight = Original Frozen Weight + (LoRA_B × LoRA_A) × scaling
finetuned_model.eval()      # Switches the model from training mode to inference mode by disabling dropout (randomly switching off neurons during training to prevent overfitting). During inference you want every neuron active for consistent answers.

print("Saving merged model...")
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_SOURCE)   # tokenizer saved together with the adapter
finetuned_model.save_pretrained(MERGED_OUTPUT_DIR)
tokenizer.save_pretrained(MERGED_OUTPUT_DIR)

print(f"Merged model saved to: {MERGED_OUTPUT_DIR}")
print(f"Files: {sorted(os.listdir(MERGED_OUTPUT_DIR))}")

## b. Push the Merged model Files to HuggingFace Hub

After merging, push the standalone model to HuggingFace Hub so it can be loaded from anywhere without needing local files.

**Before pushing, you need a Write token:**  
Go to → **huggingface.co → Profile → Settings → Access Tokens → New Token → Type: Write**
- The repo is created under **your own** Hugging Face username (read from your token).

In [ ]:
import os

if os.path.exists(os.path.join(MERGED_OUTPUT_DIR, "model.safetensors")):
    from huggingface_hub import HfApi, create_repo, login
    from getpass import getpass

    # 1. LOGIN (SECURE INPUT) — needs a token of type "Write"
    hf_token = getpass("Enter your Hugging Face API token: ")
    login(token=hf_token)

    # 2. REPO DETAILS — the repo is created under YOUR Hugging Face username
    api = HfApi()
    repo_id = f"{api.whoami()['name']}/tinyllama-unsloth-merged"

    # 3. CREATE REPO (IF NOT EXISTS)
    print("Creating repo (if needed)...")
    create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)
    print(f"Repo ready: https://huggingface.co/{repo_id}")

    # 4. UPLOAD FILES
    print("Uploading merged model (this may take a while)...")
    api.upload_folder(folder_path=MERGED_OUTPUT_DIR, repo_id=repo_id, repo_type="model")

    print("Upload complete!")
    print(f"\nYour merged model is live at: https://huggingface.co/{repo_id}")
else:
    print('⏭️  Skipping upload: no local merged model found (they are created by the cells above, after training).')

## c. Inference by loading merged model from hugging face

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

# 1. Model Repo - HuggingFace repository containing merged model (base + LoRA weights combined)
MERGED_MODEL_ID = "arif-butt/tinyllama-unsloth-merged"

# 2. Load tokenizer - converts text to token IDs and back
# Tokenizer config (padding, special tokens) is preserved from training
tokenizer = AutoTokenizer.from_pretrained(MERGED_MODEL_ID)

# 3. Load merged model (standalone) - no Unsloth/PEFT needed after merging
# AutoModelForCausalLM automatically detects architecture (Llama, GPT, etc.)
model = AutoModelForCausalLM.from_pretrained(
    MERGED_MODEL_ID,
    dtype=torch.float16,  # use half precision for GPU efficiency (reduces memory by ~50%)
    device_map="auto"           # automatically use GPU if available, falls back to CPU
)

# 4. Create pipeline - high-level API for text generation
# Handles tokenization, model inference, and decoding in one interface
pipe = pipeline(
    task="text-generation",      # specifies this is a causal language model task
    model=model,                  # the loaded merged model
    tokenizer=tokenizer,          # tokenizer for encoding/decoding
    return_full_text=False        # only return the model-generated answer (exclude input prompt)
)

In [ ]:
# 5. Example prompt
prompt = "Q: Who is Dr. Arif Butt?\nA:"

output = pipe(
    prompt,
    max_new_tokens=150,
    do_sample=False,           # deterministic/factual output
    repetition_penalty=1.1
)

# 6. Print answer
print(output[0]["generated_text"].strip())

###  **Files Uploaded to HuggingFace Hub**

After pushing the merge model to hugging face, your repository on HuggingFace Hub will contain:

```
your-username/tinyllama-unsloth-merged/
├── model.safetensors          (~2.2 GB)  — Merged model weights (base + LoRA baked in). All neural network parameters in SafeTensors format.
├── config.json                (~713 B)   — Model architecture config: type (LlamaForCausalLM), hidden size, num layers, attention heads, vocab size.
├── tokenizer.json             (~1.84 MB) — Fast tokenizer: full vocabulary, BPE merge rules, special token mappings.
├── tokenizer_config.json      (~1.32 KB) — Tokenizer settings: class, special tokens (bos/eos/pad), padding side, chat template.
├── chat_template.jinja        (~1 KB)    — Jinja2 template for TinyLlama's chat prompt format (<|system|>, <|user|>, <|assistant|> with </s> endings).
├── generation_config.json     (~124 B)   — Default generation parameters (eos_token_id, bos_token_id).
└── special_tokens_map.json    (~437 B)   — Maps special token names to their string values.
```

These files are everything needed to load and run the model from any machine

# <span style='background :lightgreen' >12. Generate GGUF, Push it to HuggingFace Hub and do local Inference via Ollama and llama.cpp</span>
## a. Generate GGUF

- **GPT-Generated Unified Format (GGUF)** is a modern, unified model file format designed for efficient local inference, especially for *LLaMA-family* of models using tools like **llama.cpp** and **Ollama**.
    - *GGUF File Container:* The outer wrapper that holds all model components in a single binary file, eliminating the need for multi-file setups and ensuring consistency between model components.
    - *Metadata Layer:* This layer stores model configuration and descriptive information, including model architecture and dimensions, context length, special tokens (BOS, EOS, PAD) and inference-related parameters.
    - *Tokenization Layer:* This layer defines how raw text is converted into tokens.
    - *Quantization Layer:* This layer describes how weights are compressed using different quantization formats like Q4, and Q8. This enables running large models on limited hardware.
    - *Model Weights and Tensors:* This layer contains the actual neural network parameters stored as optimized tensors.

**How the conversion works (next two cells):**
- `git clone --depth=1 https://github.com/ggml-org/llama.cpp` downloads llama.cpp, which contains the official converter `convert_hf_to_gguf.py` and its `gguf-py` package (no compilation needed — the converter is plain Python and uses the libraries already in our environment).
- `convert_hf_to_gguf.py` converts the merged Hugging Face model in `MERGED_OUTPUT_DIR` into a single GGUF file (`tinyllama-unsloth-chat.Q8_0.gguf`), quantized to `q8_0`.
- `upload_folder()` from `huggingface_hub` then uploads the GGUF file to your own Hugging Face repo so it can be downloaded and run locally.

**Why `q8_0`?**
- 8-bit quantization — almost zero accuracy loss compared to fp16
- TinyLlama Q8_0 ≈ 1.1 GB — runs easily on any laptop CPU
- Best balance of size and accuracy for a domain-specific Q&A model

| GGUF Format | Size | Accuracy |
|---|---|---|
| `q8_0` | ~1.1 GB | Almost no loss |
| `q5_k_m` | ~0.7 GB | Very small loss |
| `q4_k_m` | ~0.6 GB | Slight loss |

> `convert_hf_to_gguf.py` can write `f32`, `f16`, `bf16` and `q8_0` directly. The `q5_k_m` / `q4_k_m` formats need the compiled `llama-quantize` tool from llama.cpp.

In [ ]:
# Download llama.cpp (only its Python converter is used, nothing is compiled)
import os, subprocess

LLAMA_CPP_DIR = "./llama.cpp"
if not os.path.exists(os.path.join(LLAMA_CPP_DIR, "convert_hf_to_gguf.py")):
    subprocess.run(["git", "clone", "--depth=1", "https://github.com/ggml-org/llama.cpp", LLAMA_CPP_DIR], check=True)
print("llama.cpp converter ready:", os.path.join(LLAMA_CPP_DIR, "convert_hf_to_gguf.py"))

In [ ]:
import os, sys, subprocess

GGUF_DIR  = "./tinyllama-unsloth-gguf"                                     # output folder for the GGUF file
GGUF_PATH = os.path.join(GGUF_DIR, "tinyllama-unsloth-chat.Q8_0.gguf")
os.makedirs(GGUF_DIR, exist_ok=True)

if os.path.exists(os.path.join(MERGED_OUTPUT_DIR, "config.json")):
    print(f"Converting {MERGED_OUTPUT_DIR} -> {GGUF_PATH} (q8_0) ...")
    subprocess.run([
        sys.executable, os.path.join(LLAMA_CPP_DIR, "convert_hf_to_gguf.py"),
        MERGED_OUTPUT_DIR,                # merged Hugging Face model (config.json + model.safetensors + tokenizer)
        "--outfile", GGUF_PATH,           # single output GGUF file
        "--outtype", "q8_0",              # 8-bit quantization
    ], check=True)
    print(f"GGUF conversion done! {GGUF_PATH} ({os.path.getsize(GGUF_PATH) / 1e9:.2f} GB)")
else:
    print('⏭️  Skipping conversion: no merged model found in MERGED_OUTPUT_DIR (run the merge cell above first).')

## b. Push GGUF file on Hugging Face Hub

In [ ]:
import os

if os.path.exists(os.path.join(GGUF_DIR, "tinyllama-unsloth-chat.Q8_0.gguf")):
    from huggingface_hub import HfApi, create_repo, login
    from getpass import getpass

    # 1. LOGIN (SECURE INPUT) — needs a token of type "Write"
    hf_token = getpass("Enter your Hugging Face API token: ")
    login(token=hf_token)

    # 2. REPO DETAILS — the repo is created under YOUR Hugging Face username
    api = HfApi()
    repo_id = f"{api.whoami()['name']}/tinyllama-unsloth-gguf"

    # 3. CREATE REPO (IF NOT EXISTS)
    print("Creating repo (if needed)...")
    create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)
    print(f"Repo ready: https://huggingface.co/{repo_id}")

    # 4. UPLOAD FILES
    print("Uploading GGUF file (this may take a while)...")
    api.upload_folder(folder_path=GGUF_DIR, repo_id=repo_id, repo_type="model")

    print("Upload complete!")
    print(f"\nYour GGUF file is live at: https://huggingface.co/{repo_id}")
else:
    print('⏭️  Skipping upload: no local GGUF file found (they are created by the cells above, after training).')

## c. Inference by using GGUF from Hugging Face Hub using Ollama or llama.cpp
<h1 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">Ollama lets you download models from Ollama Hub or the Hugging Face Hub.</div></h1>

### Step 1: Download and Install Ollama
- Download Ollama for your Mac, Linux or Windows machine by visiting: https://ollama.com/download
- To check if ollama has been installed on your machine, open a terminal and give the command `ollama`:
```
Available Commands:
  serve       Start ollama
  create      Create a model
  show        Show information for a model
  run         Run a model
  stop        Stop a running model
  pull        Pull a model from a registry
  push        Push a model to a registry
  signin      Sign in to ollama.com
  signout     Sign out from ollama.com
  list        List models
  ps          List running models
  cp          Copy a model
  rm          Remove a model
  launch      Launch an integration with Ollama
  help        Help about any command
Use "ollama [command] --help" for more information about a specific command.
```
- To ensure Ollama application is running in the background, just open a browser and go to http://localhost:11434/ and it will display a message saying **"ollama is running"**. If this is not the case, then on the terminal give the command `ollama serve` and try again :)

### Step 2: Download the GGUF version of your Fine-Tuned model
- Ollama only runs models in the GGUF format because it is built on llama.cpp which reads GGUF files for efficient inference
- Go to **https://huggingface.co/arif-butt/tinyllama-unsloth-gguf**, click ‘Files and Versions’ on the model page, and then click the Download button on **https://huggingface.co/arif-butt/tinyllama-unsloth-gguf/blob/main/tinyllama-unsloth-chat.Q8_0.gguf** to download the model on your local machine
- If you did not convert your own GGUF above, the cell after the next one downloads this published file for you (into `./tinyllama-unsloth-gguf/`).

### Step 3: Create a Modelfile
- A **Modelfile** in Ollama is a small text configuration file that tells Ollama how to use a specific model.
- Think of it as a bridge between the GGUF model file and Ollama’s chat interface, it defines which model to load and how the conversation should behave.
- **Why is it needed?** Ollama itself cannot automatically know:
  - Which GGUF model to load.
  - How to handle prompts, messages, and system instructions.
  - What generation parameters to use (temperature, top-p, etc.).
- The Modelfile encapsulates all this information so you can run your model with consistent behavior.
- It makes customizing models easier without modifying the GGUF file itself.
- The next cell writes the `Modelfile` for you, right next to the GGUF file (it uses the GGUF you converted above, or downloads the published one from the Hub if you have none). Its content looks like this:
```
FROM <path-to-your>/tinyllama-unsloth-chat.Q8_0.gguf
SYSTEM """You are a helpful assistant."""

TEMPLATE """<|system|>
{{ .System }}</s>
<|user|>
{{ .Prompt }}</s>
<|assistant|>
"""

PARAMETER stop "<|system|>"
PARAMETER stop "<|user|>"
PARAMETER stop "<|assistant|>"
PARAMETER stop "</s>"
PARAMETER temperature 0.7
PARAMETER num_ctx 256
```

In [ ]:
import os, shutil, subprocess
from huggingface_hub import hf_hub_download

GGUF_DIR  = globals().get("GGUF_DIR", "./tinyllama-unsloth-gguf")
GGUF_PATH = os.path.join(GGUF_DIR, "tinyllama-unsloth-chat.Q8_0.gguf")
if not os.path.exists(GGUF_PATH):                     # no local GGUF: download the published one
    print("No local GGUF found, downloading arif-butt/tinyllama-unsloth-gguf from the Hugging Face Hub ...")
    GGUF_PATH = hf_hub_download("arif-butt/tinyllama-unsloth-gguf", "tinyllama-unsloth-chat.Q8_0.gguf", local_dir=GGUF_DIR)

MODELFILE_PATH = os.path.join(GGUF_DIR, "Modelfile")
modelfile = '''FROM {gguf_path}
SYSTEM """You are a helpful assistant."""

TEMPLATE """<|system|>
{{ .System }}</s>
<|user|>
{{ .Prompt }}</s>
<|assistant|>
"""

PARAMETER stop "<|system|>"
PARAMETER stop "<|user|>"
PARAMETER stop "<|assistant|>"
PARAMETER stop "</s>"
PARAMETER temperature 0.7
PARAMETER num_ctx 256
'''.replace("{gguf_path}", os.path.abspath(GGUF_PATH))

with open(MODELFILE_PATH, "w", encoding="utf-8") as f:
    f.write(modelfile)
print(f"Modelfile written to {MODELFILE_PATH}:\n")
print(modelfile)

def ollama(*args):
    # Run an `ollama` CLI command if Ollama is installed, otherwise explain how to get it
    if shutil.which("ollama") is None:
        print("⏭️  Ollama is not installed (or not on PATH): install it from https://ollama.com/download and run this cell again.")
        return
    result = subprocess.run(["ollama", *args], capture_output=True, text=True, encoding="utf-8", errors="replace")
    print(result.stdout.strip() or result.stderr.strip())

In [ ]:
ollama("create", "tinyllama-ft-unsloth", "-f", MODELFILE_PATH)

In [ ]:
ollama("list")

### Step 5 - Run your new Model via CLI

In [ ]:
ollama("run", "tinyllama-ft-unsloth", "What is the email address of Dr. Arif Butt?")

## Accessing via llama.cpp
Install llama.cpp from https://github.com/ggml-org/llama.cpp/releases (or `brew install llama.cpp` on Mac), then:
#### Via Command Line Terminal:
```bash
$ llama-cli -m ./tinyllama-unsloth-gguf/tinyllama-unsloth-chat.Q8_0.gguf
```
<div style="text-align:center;">
    <img src="../images/llamacpp-ft.png"
         style="max-width:1000px; width:100%; height:auto; display:inline-block;">
</div>

#### Via Browser:
- You can start the llama-server and load the model. The embedded server often exposes a lightweight Web UI at `http://localhost:PORT` where you can interact with the model via browser.
```bash
$ llama-server -m ./tinyllama-unsloth-gguf/tinyllama-unsloth-chat.Q8_0.gguf   --port 8080
```
<div style="text-align:center;">
    <img src="../images/llamacpp-ft2.png"
         style="max-width:1000px; width:100%; height:auto; display:inline-block;">
</div>

## d. (Alternative) Export GGUF with Unsloth's built-in `save_pretrained_gguf()`

`save_pretrained_gguf()` exports the merged model to GGUF format in one call, and `push_to_hub_gguf()` uploads the GGUF file directly to the HuggingFace Hub.

> Unsloth builds llama.cpp from source behind the scenes, so this needs a C++ build toolchain — it works on Colab / Linux, but usually not on a standard Windows setup. The cells are kept commented out for reference; the llama.cpp converter used above works everywhere. Run them right after training, while the Unsloth `model` is still in memory.

In [ ]:
#from huggingface_hub import login
#login(token="hf_yourHFtoken", add_to_git_credential=True)

In [ ]:
#print("Exporting to GGUF format (Q8_0)...")
#model.save_pretrained_gguf(
#    "TinyLlama_gguf-Finetune--Unsloth-DrArif",
#    tokenizer,
#    quantization_method="q8_0",
#)
#print("GGUF model saved → TinyLlama_gguf-Finetune--Unsloth-DrArif/")

#HF_USERNAME = "your-hf-username"       # replace with your HuggingFace username
#MERGED_REPO = f"{HF_USERNAME}/TinyLlama-Finetune-Unsloth-DrArif"

#print("Pushing GGUF to HuggingFace Hub...")
#model.push_to_hub_gguf(
#    MERGED_REPO,
#    tokenizer,
#    quantization_method="q8_0",
#)
#print(f"GGUF uploaded → https://huggingface.co/{MERGED_REPO}")

# <span style='background :lightgreen' >13. Inference with the Merged Model from the Hub: `pipeline()` vs `generate()`</span>

Now that the model is on HuggingFace Hub, we load it directly using `transformers` and run inference using two methods:

| Method | Steps |
|---|---|
| **Method 1** | `AutoTokenizer` → `AutoModelForCausalLM` → `pipeline()` |
| **Method 2** | `AutoTokenizer` → `AutoModelForCausalLM` → `apply_chat_template()` → `generate()` |

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MERGED_REPO = "arif-butt/tinyllama-unsloth-merged"

SYSTEM_PROMPT = "You are a helpful assistant."   # use the same system prompt the model was fine-tuned with

print("Loading merged model from HuggingFace Hub...")
hf_tokenizer = AutoTokenizer.from_pretrained(MERGED_REPO)
hf_model = AutoModelForCausalLM.from_pretrained(
    MERGED_REPO,
    dtype=torch.float16,
    device_map="auto",
)
hf_model.eval()
print("Model loaded successfully!")

### **Method 1: `AutoTokenizer` → `AutoModelForCausalLM` → `pipeline()`**

The already-loaded model and tokenizer are passed directly into `pipeline()`.  
Pipeline handles prompt formatting, tokenization, generation, and decoding internally.

In [ ]:
from transformers import pipeline

# Step 3 — Pipeline
pipe = pipeline(
    "text-generation",
    model=hf_model,
    tokenizer=hf_tokenizer,
)

# Step 4 — Chat function
def chat(user_question, max_new_tokens=256, temperature=0.7):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_question},
    ]
    result = pipe(
        messages,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=0.9,
        do_sample=True,
        repetition_penalty=1.1,
    )
    return result[0]["generated_text"][-1]["content"].strip()

# Test
test_questions = [
    "Who is Dr. Muhammad Arif Butt?",
    "What subjects does Dr. Arif Butt teach?",
    "Give me the URL of Dr. Arif Butt's YouTube channel.",
]
for question in test_questions:
    print(f"Question : {question}")
    print(f"Answer   : {chat(question)}")
    print("─" * 60)

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question : Who is Dr. Muhammad Arif Butt?


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer   : Dr. Muhammad Arif Butt is an accomplished Assistant Professor at the Department of Data Science, University of the Punjab (PU), Lahore, Pakistan. He has over 33 years of experience in teaching and management, with expertise in cybersecurity, operating systems, and artificial intelligence.
────────────────────────────────────────────────────────────
Question : What subjects does Dr. Arif Butt teach?


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer   : Dr. Muhammad Arif Butt teaches embedded and real-time operating systems, system programming, cybersecurity, and artificial intelligence at the University of the Punjab.
────────────────────────────────────────────────────────────
Question : Give me the URL of Dr. Arif Butt's YouTube channel.
Answer   : Dr. Arif Butt's YouTube channel 'Learn With Arif' can be found at https://www.youtube.com/c/LearnWithArif/playlists.
────────────────────────────────────────────────────────────


### **Method 2: `AutoTokenizer` → `AutoModelForCausalLM` → `apply_chat_template()` → `generate()`**

Full manual control over the inference pipeline:
1. `apply_chat_template()` — formats the conversation into TinyLlama's chat-template string
2. `tokenizer()` — converts the string to token IDs
3. `model.generate()` — generates new tokens with full control over sampling parameters
4. `tokenizer.decode()` — converts only the new tokens back to readable text

In [ ]:
# Step 3 & 4 — apply_chat_template + generate
def chat(user_question, max_new_tokens=256, temperature=0.7):

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_question},
    ]

    # Step 3 — apply_chat_template
    prompt = hf_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,     # <|assistant|> token add at the end
    )

    # Tokenize
    inputs = hf_tokenizer(prompt, return_tensors="pt").to(hf_model.device)

    # Step 4 — generate()
    output_ids = hf_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=0.9,
        do_sample=True,
        repetition_penalty=1.1,
        pad_token_id=hf_tokenizer.eos_token_id,
    )

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    response = hf_tokenizer.decode(new_tokens, skip_special_tokens=True)
    return response.strip()

# Test
test_questions = [
    "Who is Dr. Muhammad Arif Butt?",
    "What subjects does Dr. Arif Butt teach?",
    "Give me the URL of Dr. Arif Butt's YouTube channel.",
]
for question in test_questions:
    print(f"Question : {question}")
    print(f"Answer   : {chat(question)}")
    print("─" * 60)

Question : Who is Dr. Muhammad Arif Butt?
Answer   : Dr. Muhammad Arif Butt is an accomplished Assistant Professor at the Department of Data Science, University of the Punjab (PU), Lahore, Pakistan. He has over 33 years of experience in teaching and management, with expertise in cybersecurity, operating systems, and artificial intelligence.
────────────────────────────────────────────────────────────
Question : What subjects does Dr. Arif Butt teach?
Answer   : Dr. Muhammad Arif Butt teaches embedded and real-time operating systems, system programming, cybersecurity, and artificial intelligence at the University of the Punjab.
────────────────────────────────────────────────────────────
Question : Give me the URL of Dr. Arif Butt's YouTube channel.
Answer   : Dr. Arif Butt's YouTube channel 'Learn With Arif' can be found at https://www.youtube.com/c/LearnWithArif/playlists.
────────────────────────────────────────────────────────────


# <span style='background :lightgreen' >14. Windows Quick Guide: Ollama and llama.cpp</span>

A short step-by-step version of section 12 for Windows users. The GGUF model is on the HuggingFace Hub, and we can run it locally using two methods:

| Method | Best for |
|---|---|
| **Ollama** | Simple setup, runs as a local server, easiest to use |
| **llama.cpp** | Maximum control, runs on CPU or GPU, no server needed |

### **First: Download the GGUF Model**
1. Go to: **https://huggingface.co/arif-butt/tinyllama-unsloth-gguf**
2. Download file: `tinyllama-unsloth-chat.Q8_0.gguf`
3. Save it in a folder e.g. `C:\Models\`

## **Method 1: Ollama**

### Step 1: Install Ollama
- Go to **https://ollama.com/download**
- Download and install for your OS

### Step 2: Create Modelfile
Create a file named `Modelfile` (no extension) in `C:\Models\` with this content:

```
FROM tinyllama-unsloth-chat.Q8_0.gguf

SYSTEM """You are a helpful assistant."""

TEMPLATE """<|system|>
{{ .System }}</s>
<|user|>
{{ .Prompt }}</s>
<|assistant|>
"""

PARAMETER stop "<|system|>"
PARAMETER stop "<|user|>"
PARAMETER stop "<|assistant|>"
PARAMETER stop "</s>"
PARAMETER temperature 0.7
PARAMETER num_ctx 256
```

### Step 3: Open CMD and run
```
cd C:\Models
ollama create tinyllama-finetuned -f Modelfile
ollama run tinyllama-finetuned
```

### Step 4: Chat!
Type your question in CMD and press Enter.

In [ ]:
from IPython.display import Image, display

img = Image(url="https://huggingface.co/Abeehaaa/TinyLlama-Finetune-Unsloth-DrArif/resolve/main/finetune1.JPG", width=1000)
display(img)

## **Method 2: llama.cpp**

### Step 1: Download llama.cpp
- Go to: **https://github.com/ggml-org/llama.cpp/releases** (latest release)
- No NVIDIA GPU → download: `llama-<build>-bin-win-cpu-x64.zip`
- NVIDIA GPU → download: `llama-<build>-bin-win-cuda-12.4-x64.zip` **and** `cudart-llama-bin-win-cuda-12.4-x64.zip` (the CUDA runtime DLLs), and extract both into the same folder
- Extract to `C:\llama.cpp\`

### Step 2: Open CMD and run
```
cd C:\llama.cpp
llama-cli.exe -m "C:\Models\tinyllama-unsloth-chat.Q8_0.gguf" -p "Who is Arif Butt?" -n 256
```

In [ ]:
from IPython.display import Image, display

img = Image(url="https://huggingface.co/Abeehaaa/TinyLlama-Finetune-Unsloth-DrArif/resolve/main/finetune2.JPG", width=1000)
display(img)

---